# Feature Engineering — Booking Cancellation Prediction

Joins bookings with guest + property attributes and derives pre-stay
features. EXCLUDES post-stay leakage (status, total_amount realised).

**Reads:** `silver_bookings`, `silver_guests`, `silver_properties`  **Writes:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
bk = spark.read.table('silver_bookings')
g = spark.read.table('silver_guests')
p = spark.read.table('silver_properties')
print(f'bookings={bk.count():,} guests={g.count():,} properties={p.count():,}')

required = {'booking_id', 'guest_id', 'property_id', 'is_cancelled', 'lead_time_days'}
missing = required - set(bk.columns)
if missing:
    raise ValueError(f'silver_bookings missing columns {sorted(missing)}. Regenerate data and rerun bronze/silver.')

In [ ]:
# Join attributes + select pre-stay features. EXCLUDE leakage (status, total_amount).
ml_features = (
    bk.select(
        'booking_id', 'guest_id', 'property_id',
        'nights', 'room_type', 'channel', 'meal_plan', 'room_rate',
        'lead_time_days', 'is_refundable', 'loyalty_tier',
        col('is_cancelled').alias('had_cancel'),
    )
    .join(g.select('guest_id', 'region', 'age_group', 'total_stays', 'total_spend'),
          'guest_id', 'left')
    .join(p.select('property_id', 'property_type', 'star_rating', 'room_count'),
          'property_id', 'left')
    .na.fill(0)
    .na.fill('unknown', subset=['room_type', 'channel', 'meal_plan', 'loyalty_tier',
                                'region', 'age_group', 'property_type'])
    .withColumn('feature_timestamp', current_timestamp())
)

total_rows = ml_features.count()
positive_rows = ml_features.filter(col('had_cancel') == 1).count()
cancel_rate = (positive_rows / total_rows * 100) if total_rows else 0.0

if total_rows < 1000 or positive_rows < max(10, int(total_rows * 0.01)):
    raise ValueError(
        f'Label quality check failed: only {positive_rows}/{total_rows} cancelled rows '
        f'({cancel_rate:.2f}%). Check is_cancelled typing and source data.'
    )

ml_features.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_features')
print(f'Gold ML features written: {total_rows:,} rows | cancellation rate {cancel_rate:.1f}%')

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')